In [ ]:
# Install required packages (run this cell in Colab)
!pip install gitpython faiss-cpu sentence-transformers transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 40.3 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling 

In [ ]:
repo_url = "https://github.com/mcauley-penney/visual-whitespace.nvim.git"

In [ ]:
import os
import faiss
import numpy as np
import torch
from git import Repo
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# -------------------------------
# 1. Clone GitHub Repository
# -------------------------------
def clone_repo(repo_url, clone_dir="repo"):
    """
    Clone a GitHub repository if it hasn't been cloned already.
    """
    if not os.path.exists(clone_dir):
        print(f"Cloning repository from {repo_url} ...")
        Repo.clone_from(repo_url, clone_dir)
        print("Clone complete.")
    else:
        print("Repository already cloned.")
    return clone_dir

# Replace with the GitHub repo URL you want to analyze.
repo_dir = clone_repo(repo_url, clone_dir="repo")

# -------------------------------
# 2. Load and Process Files from the Repo
# -------------------------------
def is_binary_file(file_path, blocksize=1024):
    """
    Check if a file is binary by reading its first block and looking for null bytes.
    """
    try:
        with open(file_path, 'rb') as f:
            chunk = f.read(blocksize)
            if b'\0' in chunk:
                return True
    except Exception as e:
        print(f"Error reading {file_path} in binary mode: {e}")
        return True  # If we can't read it, assume binary
    return False

def load_files(repo_dir, file_extensions=None):
    """
    Recursively load text from files in a repository.

    If file_extensions is provided (as a list of strings), only files with those extensions
    will be loaded. If file_extensions is None, it will attempt to load all files that are
    not binary.
    """
    texts = []
    for root, dirs, files in os.walk(repo_dir):
        for file in files:
            # If file_extensions is specified, skip files that don't match.
            if file_extensions and not any(file.endswith(ext) for ext in file_extensions):
                continue
            file_path = os.path.join(root, file)
            # Skip binary files.
            if is_binary_file(file_path):
                continue
            try:
                with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                    texts.append(f.read())
            except Exception as e:
                print(f"Error reading {file_path}: {e}")
    return texts

all_files_text = load_files(repo_dir)
print(f"Loaded {len(all_files_text)} files from the repository.")

# -------------------------------
# 3. Chunk the Text
# -------------------------------
def chunk_text(text, chunk_size=300):
    """
    Split text into chunks of ~chunk_size words.
    """
    words = text.split()
    return [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

all_chunks = []
for file_text in all_files_text:
    chunks = chunk_text(file_text, chunk_size=300)
    all_chunks.extend(chunks)
print(f"Created {len(all_chunks)} text chunks.")

# -------------------------------
# 4. Vectorize Chunks with SentenceTransformer and Build FAISS Index
# -------------------------------
# Load a free open-source sentence embedding model
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Compute embeddings for each chunk
print("Computing embeddings for text chunks...")
chunk_embeddings = embedder.encode(all_chunks, convert_to_numpy=True)
embedding_dim = chunk_embeddings.shape[1]

# Create and populate the FAISS index
index = faiss.IndexFlatL2(embedding_dim)
index.add(chunk_embeddings)
print("FAISS index created.")

# -------------------------------
# 5. Set Up the LLM for Response Generation
# -------------------------------
# Load a free open-source LLM (using a small Flan-T5 model for demonstration)
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"LLM loaded on {device}.")

# -------------------------------
# 6. Define Retrieval and Answer Generation Functions
# -------------------------------
def retrieve_context(query, k=5):
    """
    Given a query, retrieve the k most similar text chunks.
    """
    query_embedding = embedder.encode(query, convert_to_numpy=True)
    distances, indices = index.search(np.array([query_embedding]), k)
    retrieved = [all_chunks[i] for i in indices[0]]
    return "\n".join(retrieved)

def generate_answer(query):
    """
    Generate an answer by retrieving context and then using the LLM.
    """
    context = retrieve_context(query, k=5)
    # Build a prompt that supplies context and the question to the LLM.
    prompt = (
        "You are an assistant that can answer questions about a GitHub repository's code and documentation. "
        "Use the context provided to answer the question accurately.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\nAnswer:"
    )
    inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    outputs = model.generate(**inputs, max_length=200, num_beams=5, early_stopping=True)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer

Cloning repository from https://github.com/mcauley-penney/visual-whitespace.nvim.git ...
Clone complete.
Loaded 25 files from the repository.
Created 33 text chunks.


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

LLM loaded on cpu.


In [ ]:
# -------------------------------
# 7. Chat Loop
# -------------------------------
print("\nChatbot is ready! Type your questions about the repository (type 'exit' to quit).")
while True:
    user_query = input("You: ")
    if user_query.lower() in ['exit', 'quit']:
        print("Exiting chatbot. Goodbye!")
        break
    response = generate_answer(user_query)
    print("\nBot:", response, "\n")


Chatbot is ready! Type your questions about the repository (type 'exit' to quit).

Bot: #!/bin/sh # # An example hook script to block unannotated tags from entering the # repository. By default they won't be. # hooks.allowunannotated # This boolean sets whether unannotated tags will be allowed into the # repository. By default they won't be. # hooks.allowdeletetag # This boolean sets whether deleting tags will be allowed in the # repository. By default they won't be. # hooks.allowdeletebranch # This boolean sets whether remotely creating branches will be denied # in the # repository. By default this is allowed. # # --- Command line refname="$1" oldrev="$2" newrev="$3" # # Config # ------ # hooks.allowunannotated # This boolean sets whether unannotated 


Bot: ## Installation and configuration To install the plugin with the default settings using Lazy: lua  'mcauley-penney/visual-whitespace.nvim', config = true   visual-whitespace. 


Bot: #  visual-whitespace.nvim Reveal whitespace ch

KeyboardInterrupt: Interrupted by user